# Fine-tuning DistilBERT for Humor: 256 tokens (Colab)

Identical to the 128-token run **except `MAX_LENGTH = 256`**. Everything else
(epochs, seed, LR, data) is held constant so the comparison is clean: any
change in Spearman is attributable to sequence length alone. Colab version of the 256-token experiment.

- 128-token result to beat: **test Spearman 0.4038**
- New model pushed to a **separate** HF repo, so the 128 model stays intact.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

In [ ]:
# CONFIG
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/humor-intelligence.git"
HF_MODEL_REPO   = "YOUR_HF_USERNAME/humor-intelligence-distilbert-256"

MODEL_NAME  = "distilbert-base-uncased"
MAX_LENGTH  = 256
EPOCHS      = 2
BATCH_SIZE  = 32
LR          = 2e-5
SEED        = 42
CKPT_DIR = "humor-intelligence-distilbert-256"

In [ ]:
# 1. GPU check
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (!)")

In [ ]:
# 2. Install HF stack
!pip install -q "transformers>=4.40" "datasets>=2.19" "accelerate>=0.30" scipy

In [ ]:
# 3. Clone repo & rebuild cleaned data on Colab
import os, shutil
if os.path.exists("humor-intelligence"):
    shutil.rmtree("humor-intelligence")
!git clone -q $GITHUB_REPO_URL
%cd humor-intelligence
!python src/data.py
%cd ..

In [ ]:
# 4. Load cleaned splits
import pandas as pd
BASE = "humor-intelligence/data/processed"
train = pd.read_csv(f"{BASE}/train.csv")
dev   = pd.read_csv(f"{BASE}/dev.csv")
test  = pd.read_csv(f"{BASE}/test.csv")
print(f"train {len(train):,} | dev {len(dev):,} | test {len(test):,}")

In [ ]:
# 5. Tokenize at MAX_LENGTH=256

from datasets import Dataset, Value
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_ds(df):
    ds = Dataset.from_pandas(
        df[["joke", "score"]].rename(columns={"score": "labels"}),
        preserve_index=False,
    )
    ds = ds.map(
        lambda b: tokenizer(b["joke"], truncation=True, max_length=MAX_LENGTH),
        batched=True,
    )
    ds = ds.cast_column("labels", Value("float32"))
    return ds

train_ds, dev_ds, test_ds = make_ds(train), make_ds(dev), make_ds(test)
print("label dtype:", train_ds.features["labels"].dtype)

In [ ]:
# 6. Regression model & Spearman metric

import numpy as np
from scipy.stats import spearmanr, pearsonr
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, problem_type="regression"
)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.asarray(preds).squeeze()
    return {"spearman": float(spearmanr(labels, preds).correlation),
            "pearson":  float(pearsonr(labels, preds)[0])}

In [ ]:
# 7. Train
from transformers import (TrainingArguments, Trainer, DataCollatorWithPadding)
from transformers.trainer_utils import get_last_checkpoint

collator = DataCollatorWithPadding(tokenizer)

args = TrainingArguments(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    seed=SEED,
    fp16=True,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    report_to="none",
)

trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=dev_ds,
                  compute_metrics=compute_metrics, data_collator=collator)

last_ckpt = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
if last_ckpt:
    print("Resuming from", last_ckpt)
trainer.train(resume_from_checkpoint=last_ckpt)

In [ ]:
# 8. Test-set evaluation
test_metrics = trainer.evaluate(test_ds)
print("256-token TEST Spearman:", round(test_metrics["eval_spearman"], 4))
print("256-token TEST Pearson :", round(test_metrics["eval_pearson"], 4))
print("\n128-token was 0.4038")

In [ ]:
# 9. Push to the NEW HF repo
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 10. Push to the HF repo
trainer.push_to_hub("End of training")
tokenizer.push_to_hub(HF_MODEL_REPO)
print("Pushed to https://huggingface.co/" + HF_MODEL_REPO)